# 01 — Data Collection
**Project:** Machine-Learning-Based Sector Rotation Regime Discovery — Indian Equity Market  
**Phase:** 1 — Raw Market Data  
**Output:** `data/raw/market/` and `data/raw/sector_indices/`

## 1. Setup

In [2]:
import sys
import os
from pathlib import Path

import pandas as pd

# Ensure the project root is on sys.path so src/ is importable
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    VERIFIED_TICKERS,
    DEFAULT_START,
    download_data,
    save_raw_data,
    load_data,
    validate_downloaded_data,
    verify_ticker,
)

RAW_BASE = PROJECT_ROOT / "data" / "raw"
print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {RAW_BASE}")

Project root : c:\Users\vighn\OneDrive\Desktop\sector rotation regime discovery
Raw data dir : c:\Users\vighn\OneDrive\Desktop\sector rotation regime discovery\data\raw


## 2. Instruments

The research universe covers 2 market-wide and 6 sectoral NSE indices, all sourced from Yahoo Finance.

In [4]:
# Build an instruments table from the verified ticker registry in src/data_loader.py
rows = []
for key, meta in VERIFIED_TICKERS.items():
    out_path = RAW_BASE / meta["subdir"] / meta["output_file"]
    rows.append({
        "Instrument" : meta["label"],
        "Category"   : meta["category"],
        "Yahoo Ticker": meta["ticker"],
        "Output File" : meta["output_file"],
        "File Exists" : out_path.exists(),
    })

instruments_df = pd.DataFrame(rows)
instruments_df

,Instrument,Category,Yahoo Ticker,Output File,File Exists
0,NIFTY 50,Market,^NSEI,nifty50.csv,True
1,India VIX,Market,^INDIAVIX,india_vix.csv,True
2,NIFTY IT,Sector,^CNXIT,nifty_it.csv,True
3,NIFTY Bank,Sector,^NSEBANK,nifty_bank.csv,True
4,NIFTY Auto,Sector,^CNXAUTO,nifty_auto.csv,True
5,NIFTY Pharma,Sector,^CNXPHARMA,nifty_pharma.csv,True
6,NIFTY FMCG,Sector,^CNXFMCG,nifty_fmcg.csv,True
7,NIFTY Metal,Sector,^CNXMETAL,nifty_metal.csv,True


## 3. Date Range

In [5]:
START_DATE = DEFAULT_START   # "2019-01-01"
END_DATE   = None            # None => download to the latest available session

print(f"Research start : {START_DATE}")
print(f"Research end   : {'latest available' if END_DATE is None else END_DATE}")

Research start : 2019-01-01
Research end   : latest available


## 4. Ticker Verification

Quick test against a short 2024 window to confirm all Yahoo Finance tickers are live before committing to the full download.

In [6]:
verification_rows = []
for key, meta in VERIFIED_TICKERS.items():
    r = verify_ticker(
        ticker=meta["ticker"],
        instrument_label=meta["label"],
        test_start="2024-01-01",
        test_end="2024-06-30",
    )
    verification_rows.append({
        "Instrument" : r["instrument"],
        "Ticker"     : r["ticker"],
        "Status"     : r["status"],
        "Rows (test)": r["rows"],
        "First Date" : r["first_date"],
        "Last Date"  : r["last_date"],
        "Columns"    : r["columns"],
        "Error"      : r["error"],
    })

ver_df = pd.DataFrame(verification_rows)
ver_df

,Instrument,Ticker,Status,Rows (test),First Date,Last Date,Columns,Error
0,NIFTY 50,^NSEI,PASS,120,2024-01-01,2024-06-28,"[Adj Close, Close, High, Low, Open, Volume]",None
1,India VIX,^INDIAVIX,PASS,118,2024-01-02,2024-06-28,"[Adj Close, Close, High, Low, Open, Volume]",None
2,NIFTY IT,^CNXIT,PASS,120,2024-01-01,2024-06-28,"[Adj Close, Close, High, Low, Open, Volume]",None
3,NIFTY Bank,^NSEBANK,PASS,120,2024-01-01,2024-06-28,"[Adj Close, Close, High, Low, Open, Volume]",None
4,NIFTY Auto,^CNXAUTO,PASS,118,2024-01-02,2024-06-28,"[Adj Close, Close, High, Low, Open, Volume]",None
5,NIFTY Pharma,^CNXPHARMA,PASS,120,2024-01-01,2024-06-28,"[Adj Close, Close, High, Low, Open, Volume]",None
6,NIFTY FMCG,^CNXFMCG,PASS,118,2024-01-02,2024-06-28,"[Adj Close, Close, High, Low, Open, Volume]",None
7,NIFTY Metal,^CNXMETAL,PASS,118,2024-01-02,2024-06-28,"[Adj Close, Close, High, Low, Open, Volume]",None


In [7]:
failed_tickers = ver_df[ver_df["Status"] != "PASS"]
if not failed_tickers.empty:
    print("WARNING: The following tickers failed verification:")
    print(failed_tickers[["Instrument", "Ticker", "Error"]].to_string(index=False))
else:
    print("All tickers verified successfully.")

All tickers verified successfully.


## 5. Download Data

Downloads `2019-01-01` through the latest available trading session. Existing files are skipped (`overwrite=False`).

In [8]:
downloaded = {}

for key, meta in VERIFIED_TICKERS.items():
    out_path = RAW_BASE / meta["subdir"] / meta["output_file"]

    if out_path.exists():
        print(f"  SKIP  {meta['label']:<16s} — file already exists: {out_path.name}")
        continue

    df = download_data(
        ticker=meta["ticker"],
        start=START_DATE,
        end=END_DATE,
        instrument_label=meta["label"],
    )
    downloaded[key] = df

print("\nDownload step complete.")

  SKIP  NIFTY 50         — file already exists: nifty50.csv
  SKIP  India VIX        — file already exists: india_vix.csv
  SKIP  NIFTY IT         — file already exists: nifty_it.csv
  SKIP  NIFTY Bank       — file already exists: nifty_bank.csv
  SKIP  NIFTY Auto       — file already exists: nifty_auto.csv
  SKIP  NIFTY Pharma     — file already exists: nifty_pharma.csv
  SKIP  NIFTY FMCG       — file already exists: nifty_fmcg.csv
  SKIP  NIFTY Metal      — file already exists: nifty_metal.csv

Download step complete.


## 6. Save Raw Data

Saves any newly downloaded DataFrames to CSV. `overwrite=False` protects existing files.

In [9]:
for key, df in downloaded.items():
    meta = VERIFIED_TICKERS[key]
    out_path = RAW_BASE / meta["subdir"] / meta["output_file"]
    saved = save_raw_data(df, output_path=out_path, overwrite=False)
    status = "saved" if saved else "skipped (already exists)"
    print(f"  {meta['output_file']:<22s}  {status}")

print("\nSave step complete.")


Save step complete.


## 7. Data Inspection

Loads each raw CSV and displays basic structural information.

In [10]:
datasets = {}

for key, meta in VERIFIED_TICKERS.items():
    out_path = RAW_BASE / meta["subdir"] / meta["output_file"]
    if not out_path.exists():
        print(f"  MISSING: {out_path}")
        continue
    datasets[key] = load_data(out_path)

print(f"\nLoaded {len(datasets)}/8 datasets.")

2026-09-13 18:17:01  INFO      Loaded  ← c:\Users\vighn\OneDrive\Desktop\sector rotation regime discovery\data\raw\market\nifty50.csv  (1899 rows)
2026-09-13 18:17:01  INFO      Loaded  ← c:\Users\vighn\OneDrive\Desktop\sector rotation regime discovery\data\raw\market\india_vix.csv  (1883 rows)
2026-09-13 18:17:01  INFO      Loaded  ← c:\Users\vighn\OneDrive\Desktop\sector rotation regime discovery\data\raw\sector_indices\nifty_it.csv  (1899 rows)
2026-09-13 18:17:01  INFO      Loaded  ← c:\Users\vighn\OneDrive\Desktop\sector rotation regime discovery\data\raw\sector_indices\nifty_bank.csv  (1899 rows)
2026-09-13 18:17:01  INFO      Loaded  ← c:\Users\vighn\OneDrive\Desktop\sector rotation regime discovery\data\raw\sector_indices\nifty_auto.csv  (1884 rows)
2026-09-13 18:17:01  INFO      Loaded  ← c:\Users\vighn\OneDrive\Desktop\sector rotation regime discovery\data\raw\sector_indices\nifty_pharma.csv  (1898 rows)
2026-09-13 18:17:01  INFO      Loaded  ← c:\Users\vighn\OneDrive\Desktop


Loaded 8/8 datasets.


In [9]:
# Per-dataset head / tail / shape / dtypes
for key, df in datasets.items():
    meta = VERIFIED_TICKERS[key]
    print("=" * 60)
    print(f"{meta['label']}  ({meta['ticker']})")
    print(f"  shape   : {df.shape}")
    print(f"  columns : {list(df.columns)}")
    print(f"  dtypes  :\n{df.dtypes.to_string()}")
    print(f"  first   : {df.index[0].date()}")
    print(f"  last    : {df.index[-1].date()}")
    print()

NIFTY 50  (^NSEI)
  shape   : (1899, 6)
  columns : ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
  dtypes  :
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
  first   : 2019-01-02
  last    : 2026-09-11

India VIX  (^INDIAVIX)
  shape   : (1883, 6)
  columns : ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
  dtypes  :
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
  first   : 2019-01-01
  last    : 2026-09-11

NIFTY IT  (^CNXIT)
  shape   : (1899, 6)
  columns : ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
  dtypes  :
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
  first   : 2019-01-01
  last    : 2026-09-11

NIFTY Bank  (^NSEBANK)
  shape   : (1899, 6)
  columns : ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
  

In [11]:
# Quick compact coverage table
coverage_rows = []
for key, df in datasets.items():
    meta = VERIFIED_TICKERS[key]
    coverage_rows.append({
        "Instrument" : meta["label"],
        "Category"   : meta["category"],
        "Ticker"     : meta["ticker"],
        "Rows"       : len(df),
        "First Date" : df.index[0].date(),
        "Last Date"  : df.index[-1].date(),
        "Columns"    : len(df.columns),
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df

,Instrument,Category,Ticker,Rows,First Date,Last Date,Columns
0,NIFTY 50,Market,^NSEI,1899,2019-01-02,2026-09-11,6
1,India VIX,Market,^INDIAVIX,1883,2019-01-01,2026-09-11,6
2,NIFTY IT,Sector,^CNXIT,1899,2019-01-01,2026-09-11,6
3,NIFTY Bank,Sector,^NSEBANK,1899,2019-01-01,2026-09-11,6
4,NIFTY Auto,Sector,^CNXAUTO,1884,2019-01-01,2026-09-11,6
5,NIFTY Pharma,Sector,^CNXPHARMA,1898,2019-01-01,2026-09-11,6
6,NIFTY FMCG,Sector,^CNXFMCG,1883,2019-01-01,2026-09-11,6
7,NIFTY Metal,Sector,^CNXMETAL,1884,2019-01-01,2026-09-11,6


In [11]:
# Sample: head() and tail() for NIFTY 50
nifty50_df = datasets.get("nifty50")
if nifty50_df is not None:
    print("NIFTY 50 — head")
    display(nifty50_df.head())
    print("NIFTY 50 — tail")
    display(nifty50_df.tail())

NIFTY 50 — head


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2019-01-02,10792.500000,10792.500000,10895.349609,10735.049805,10868.849609,309700
2019-01-03,10672.250000,10672.250000,10814.049805,10661.250000,10796.799805,286200
2019-01-04,10727.349609,10727.349609,10741.049805,10628.650391,10699.700195,296600
2019-01-07,10771.799805,10771.799805,10835.950195,10750.150391,10804.849609,269400
2019-01-08,10802.150391,10802.150391,10818.450195,10733.250000,10786.250000,277700


NIFTY 50 — tail


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-09-07,23779.150391,23779.150391,23890.000000,23737.900391,23883.150391,205900
2026-09-08,23635.099609,23635.099609,23758.949219,23623.099609,23743.099609,230100
2026-09-09,23431.500000,23431.500000,23571.550781,23431.500000,23522.050781,352900
2026-09-10,23477.800781,23477.800781,23494.949219,23380.099609,23446.599609,253200
2026-09-11,23398.099609,23398.099609,23448.099609,23231.400391,23270.300781,293000


## 8. Basic Validation

Non-destructive checks only. Missing values and duplicates are **reported, not fixed**.

In [12]:
validation_results = []

for key, df in datasets.items():
    meta = VERIFIED_TICKERS[key]
    v = validate_downloaded_data(df, meta["label"], meta["ticker"])
    validation_results.append(v)
    print(f"[{v['status']:7s}]  {meta['label']:<16s}  rows={v['rows']:4d}  "
          f"{v['first_date']} -> {v['last_date']}  "
          f"missing={v['total_missing']}  dup={v['duplicate_dates']}  "
          f"ohlc_viol={v['ohlc_violations']}")
    for note in v["notes"]:
        if not note.startswith("All checks"):
            print(f"          NOTE: {note}")

[PASS   ]  NIFTY 50          rows=1899  2019-01-02 -> 2026-09-11  missing=0  dup=0  ohlc_viol=0
[PASS   ]  India VIX         rows=1883  2019-01-01 -> 2026-09-11  missing=0  dup=0  ohlc_viol=0
[PASS   ]  NIFTY IT          rows=1899  2019-01-01 -> 2026-09-11  missing=0  dup=0  ohlc_viol=0
[PASS   ]  NIFTY Bank        rows=1899  2019-01-01 -> 2026-09-11  missing=0  dup=0  ohlc_viol=0
[PASS   ]  NIFTY Auto        rows=1884  2019-01-01 -> 2026-09-11  missing=0  dup=0  ohlc_viol=0
[PASS   ]  NIFTY Pharma      rows=1898  2019-01-01 -> 2026-09-11  missing=0  dup=0  ohlc_viol=0
[PASS   ]  NIFTY FMCG        rows=1883  2019-01-01 -> 2026-09-11  missing=0  dup=0  ohlc_viol=0
[PASS   ]  NIFTY Metal       rows=1884  2019-01-01 -> 2026-09-11  missing=0  dup=0  ohlc_viol=0


## 9. Final Summary

Complete Phase 1 status table.

In [13]:
summary_rows = []
for v, (key, meta) in zip(validation_results, VERIFIED_TICKERS.items()):
    out_path = RAW_BASE / meta["subdir"] / meta["output_file"]
    summary_rows.append({
        "Instrument"    : meta["label"],
        "Category"      : meta["category"],
        "Ticker"        : meta["ticker"],
        "Rows"          : v["rows"],
        "First Date"    : str(v["first_date"]),
        "Last Date"     : str(v["last_date"]),
        "Missing Values": v["total_missing"],
        "Duplicate Dates": v["duplicate_dates"],
        "OHLC Violations": v["ohlc_violations"],
        "Status"        : v["status"],
        "Output File"   : meta["output_file"],
        "File Size (KB)": round(out_path.stat().st_size / 1024, 1) if out_path.exists() else "MISSING",
    })

summary_df = pd.DataFrame(summary_rows)

# Style: highlight status
def highlight_status(val):
    if val == "PASS":
        return "background-color: #d4edda; color: #155724;"
    elif val == "WARNING":
        return "background-color: #fff3cd; color: #856404;"
    elif val == "FAILED":
        return "background-color: #f8d7da; color: #721c24;"
    return ""

summary_df.style.applymap(highlight_status, subset=["Status"])

C:\Users\vighn\AppData\Local\Temp\ipykernel_13296\2333305392.py:31: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  summary_df.style.applymap(highlight_status, subset=["Status"])


,Instrument,Category,Ticker,Rows,First Date,Last Date,Missing Values,Duplicate Dates,OHLC Violations,Status,Output File,File Size (KB)
0,NIFTY 50,Market,^NSEI,1899,2019-01-02,2026-09-11,0,0,0,PASS,nifty50.csv,169.000000
1,India VIX,Market,^INDIAVIX,1883,2019-01-01,2026-09-11,0,0,0,PASS,india_vix.csv,192.800000
2,NIFTY IT,Sector,^CNXIT,1899,2019-01-01,2026-09-11,0,0,0,PASS,nifty_it.csv,161.000000
3,NIFTY Bank,Sector,^NSEBANK,1899,2019-01-01,2026-09-11,0,0,0,PASS,nifty_bank.csv,159.200000
4,NIFTY Auto,Sector,^CNXAUTO,1884,2019-01-01,2026-09-11,0,0,0,PASS,nifty_auto.csv,167.100000
5,NIFTY Pharma,Sector,^CNXPHARMA,1898,2019-01-01,2026-09-11,0,0,0,PASS,nifty_pharma.csv,173.000000
6,NIFTY FMCG,Sector,^CNXFMCG,1883,2019-01-01,2026-09-11,0,0,0,PASS,nifty_fmcg.csv,157.700000
7,NIFTY Metal,Sector,^CNXMETAL,1884,2019-01-01,2026-09-11,0,0,0,PASS,nifty_metal.csv,173.100000


In [14]:
# Print plain-text summary for copy-paste
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(summary_df.to_string(index=False))

  Instrument Category     Ticker  Rows First Date  Last Date  Missing Values  Duplicate Dates  OHLC Violations Status      Output File  File Size (KB)
    NIFTY 50   Market      ^NSEI  1899 2019-01-02 2026-09-11               0                0                0   PASS      nifty50.csv           169.0
   India VIX   Market  ^INDIAVIX  1883 2019-01-01 2026-09-11               0                0                0   PASS    india_vix.csv           192.8
    NIFTY IT   Sector     ^CNXIT  1899 2019-01-01 2026-09-11               0                0                0   PASS     nifty_it.csv           161.0
  NIFTY Bank   Sector   ^NSEBANK  1899 2019-01-01 2026-09-11               0                0                0   PASS   nifty_bank.csv           159.2
  NIFTY Auto   Sector   ^CNXAUTO  1884 2019-01-01 2026-09-11               0                0                0   PASS   nifty_auto.csv           167.1
NIFTY Pharma   Sector ^CNXPHARMA  1898 2019-01-01 2026-09-11               0                0 

In [15]:
# Confirm Phase 1 scope
pass_count    = (summary_df["Status"] == "PASS").sum()
warning_count = (summary_df["Status"] == "WARNING").sum()
failed_count  = (summary_df["Status"] == "FAILED").sum()

print("\n--- Phase 1 Completion ---")
print(f"  PASS    : {pass_count}")
print(f"  WARNING : {warning_count}")
print(f"  FAILED  : {failed_count}")
print()
print("Confirmed NOT done in this phase:")
print("  - No event data collected.")
print("  - No processed datasets created.")
print("  - No feature engineering performed.")
print("  - No regime discovery performed.")
print("  - No backtesting performed.")    


--- Phase 1 Completion ---
  PASS    : 8
  FAILED  : 0

Confirmed NOT done in this phase:
  - No event data collected.
  - No processed datasets created.
  - No feature engineering performed.
  - No regime discovery performed.
  - No backtesting performed.


In [16]:
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print(f"\n{name}")
        print(missing)
        print("\nRows containing missing values:")
        display(df[df.isna().any(axis=1)])

In [17]:
for name in ["nifty_it", "nifty_fmcg", "nifty_metal"]:
    print(f"\n{name}")
    display(datasets[name].tail(3))


nifty_it


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-09-09,28914.000000,28914.000000,29299.550781,28779.800781,29299.550781,45300
2026-09-10,28890.900391,28890.900391,29264.900391,28740.599609,28919.349609,28900
2026-09-11,28921.500000,28921.500000,29124.599609,28671.800781,28672.849609,25400



nifty_fmcg


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-09-09,45307.449219,45307.449219,45705.449219,45206.449219,45652.449219,0
2026-09-10,45183.699219,45183.699219,45476.699219,45066.449219,45342.750000,0
2026-09-11,45053.800781,45053.800781,45276.699219,44825.601562,44909.300781,0



nifty_metal


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-09-09,13391.099609,13391.099609,13437.799805,13170.299805,13180.000000,0
2026-09-10,13301.900391,13301.900391,13416.049805,13242.400391,13416.049805,0
2026-09-11,13001.650391,13001.650391,13084.099609,12886.099609,13079.849609,0
